<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
</div>

### RapidFire AI LangGraph Agent Tutorial: Trajectory-Level Metrics

This notebook demonstrates **trajectory-level evaluation** — going beyond
pass/fail to understand *how* an agent reaches its answer.

With `capture_trajectory=True`, RapidFire records every node execution,
tool call, LLM call, token count, cost, and timing. This lets you compare
configs on efficiency, reliability, and failure modes — not just accuracy.

**What you'll learn:**
- Enabling trajectory capture with `capture_trajectory=True`
- Using `compute_trajectory_metrics` for per-batch diagnostics
- Using `accumulate_trajectory_metrics` for cross-batch aggregation
- Analyzing failure modes with `analyze_failure_modes`
- Combining output-level and trajectory-level metrics in one sweep

In [ ]:
import os

from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFLangGraphAgentConfig
from rapidfireai.evals.metrics.langgraph_metrics import (
    compute_trajectory_metrics,
    accumulate_trajectory_metrics,
    analyze_failure_modes,
)

### 1. Dataset

A small multi-step Q&A dataset. Some questions need tool use (math),
others are pure knowledge — useful for comparing trajectories.

In [ ]:
from datasets import Dataset

qa_data = {
    "question": [
        "What is the capital of France?",
        "What is 25 * 4?",
        "Who wrote Romeo and Juliet?",
        "What is the boiling point of water in Celsius?",
        "What planet is closest to the Sun?",
        "What is the square root of 144?",
        "In which year did World War II end?",
        "What is the chemical symbol for gold?",
        "What is 123 * 456?",
        "How many continents are there?",
    ],
    "expected_answer": [
        "Paris",
        "100",
        "Shakespeare",
        "100",
        "Mercury",
        "12",
        "1945",
        "Au",
        "56088",
        "7",
    ],
}

dataset = Dataset.from_dict(qa_data)
print(f"Dataset size: {len(dataset)} questions")

### 2. Create Experiment

In [ ]:
experiment = Experiment(experiment_name="langgraph-trajectory-demo", mode="evals")

### 3. Tools

In [ ]:
from langchain_core.tools import tool


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"


my_tools = [calculator]

### 4. Input/Output Mappers and Combined Metrics

The key difference from the hello-world notebook: `compute_metrics_fn`
now returns **both** accuracy and trajectory diagnostics.

In [ ]:
from langchain_core.messages import HumanMessage


def input_mapper(row):
    return {"messages": [HumanMessage(content=row["question"])]}


def output_mapper(result):
    if result is None:
        return ""
    messages = result.get("messages", [])
    return messages[-1].content if messages else ""


def combined_metrics(batch):
    """Compute accuracy + trajectory diagnostics in one function."""
    outputs = batch.get("agent_output", [])
    expected = batch.get("expected_answer", [])
    n = len(outputs)

    # Output-level: fuzzy accuracy
    correct = sum(1 for pred, gt in zip(outputs, expected) if gt.lower() in pred.lower())
    metrics = {
        "accuracy": {"value": correct / n if n > 0 else 0, "is_algebraic": True},
    }

    # Trajectory-level: step count, tool calls, cost, etc.
    traj_metrics = compute_trajectory_metrics(batch)
    metrics.update(traj_metrics)

    return metrics


def combined_accumulate(metrics):
    """Accumulate both accuracy and trajectory metrics."""
    return accumulate_trajectory_metrics(metrics)

### 5. Configure with Trajectory Capture

Setting `capture_trajectory=True` tells RF to attach a `TrajectoryCallback`
that records every node execution, tool call, and LLM call.

The trajectory data is available in `batch["trajectories"]` inside
your metrics function.

In [ ]:
from langchain_openai import ChatOpenAI

config = RFGridSearch(
    {
        "pipeline": RFLangGraphAgentConfig(
            prebuilt_agent="react",
            prebuilt_agent_kwargs={
                "model": List(
                    [
                        ChatOpenAI(model="gpt-4o-mini", temperature=0),
                        ChatOpenAI(model="gpt-4o", temperature=0),
                    ]
                ),
                "tools": my_tools,
            },
            input_mapper=input_mapper,
            output_mapper=output_mapper,
            capture_trajectory=True,  # <-- This enables trajectory recording
            max_steps=15,
            timeout_seconds=60,
        ),
        "batch_size": 1,
        "compute_metrics_fn": combined_metrics,
        "accumulate_metrics_fn": combined_accumulate,
    }
)

print("Trajectory capture enabled — RF will record step-level data for each invocation")

### 6. Run Evaluation

In [ ]:
results = experiment.run_evals(
    config_group=config,
    dataset=dataset,
    num_shards=5,
    seed=42,
)

### 7. View Aggregated Results

The results now include both accuracy *and* trajectory diagnostics:
- `accuracy`: did the agent get the right answer?
- `avg_steps`: how many graph nodes were executed?
- `avg_tool_calls`: did the agent use tools efficiently?
- `avg_cost_usd`: estimated LLM API cost per question
- `timeout_rate`: what fraction of invocations timed out?

In [ ]:
import pandas as pd

rows = []
for pipeline_id, (aggregated_results, cumulative_metrics) in results.items():
    row = {"pipeline_id": pipeline_id}
    for metric_name, metric_data in cumulative_metrics.items():
        if isinstance(metric_data, dict):
            row[metric_name] = metric_data.get("value", metric_data)
        else:
            row[metric_name] = metric_data
    rows.append(row)

df = pd.DataFrame(rows)

# Display key metrics side-by-side
display_cols = [
    "pipeline_id", "model_name", "accuracy",
    "avg_steps", "avg_tool_calls", "avg_cost_usd",
    "avg_time_seconds", "timeout_rate",
]
available_cols = [c for c in display_cols if c in df.columns]
df[available_cols]

### 8. Failure Mode Analysis

For deeper analysis, use `analyze_failure_modes` on the raw trajectories.
This shows *why* a config fails — timeout, tool errors, wrong node, etc.

In [ ]:
for pipeline_id, (aggregated_results, _) in results.items():
    trajectories = aggregated_results.get("trajectories", [])
    outputs = aggregated_results.get("agent_output", [])
    expected = aggregated_results.get("expected_answer", [])

    if not trajectories:
        print(f"Pipeline {pipeline_id}: no trajectory data")
        continue

    labels = [
        gt.lower() in pred.lower()
        for pred, gt in zip(outputs, expected)
    ]

    analysis = analyze_failure_modes(trajectories, labels)
    print(f"\nPipeline {pipeline_id} failure analysis:")
    for key, value in analysis.items():
        print(f"  {key}: {value}")

### 9. End Experiment

In [ ]:
experiment.end()